## Split both mid and anom data into Train Validation and Test csv's

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Grab file
file_path_1 = "../../ais_data/mid_datav4.csv"
file_path_2 = "../../ais_data/anom_datav2.csv"
df_1 = pd.read_csv(file_path_1)
df_2 = pd.read_csv(file_path_2)
init_df = pd.concat([df_1, df_2], ignore_index=True)
print("pre dublicate voyage drop:", len(init_df))
init_df = init_df.drop_duplicates(subset=["LAT_AVG", "LON_AVG", "voyage_id"])
print("post dublicate voyage drop:", len(init_df))

pre dublicate voyage drop: 54500
post dublicate voyage drop: 52643


In [3]:
print(init_df.dtypes)

Unnamed: 0               int64
MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER             float64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
BEAM                     int64
LENGTH                   int64
time_diff              float64
new_voyage                bool
voyage_id               object
num_pings                int64
row_id                  object
channel_side            object
cog_diff

In [4]:
# Only take columns needed for TCN Neural Network
df = init_df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

# Make a datetime type column
df['TIME'] = pd.to_datetime(df['PERIOD'], errors='coerce')
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print("datetime check", df['TIME'].dtypes)

# drop all NA's
print("pre-NA drop", len(df))
df = df.dropna()
print("post-NA drop", len(df))

datetime check datetime64[ns]
pre-NA drop 52643
post-NA drop 50424


C:\Users\LAKwon\AppData\Local\Temp\ipykernel_1652\768074335.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TIME'] = pd.to_datetime(df['PERIOD'], errors='coerce')


In [5]:
# make delta time, lat, and lon columns for TCN
df['dt'] = df['TIME'].diff().dt.total_seconds()
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

# sort values by vessel and time
df = df.sort_values(["MMSI", "TIME"]).reset_index(drop=True)

# fast way to change each new voyage delta time, lat, and lon from NaN to 0
mask = df['voyage_id'] != df['voyage_id'].shift()
df.loc[mask, ['dt', 'dlat', 'dlon']] = 0

# older slower way
'''for voyage in range(0, len(df)):
    if voyage > 0 and df.loc[voyage-1, "dt"] != df.loc[voyage, "dt"]:
        df.loc[voyage, ["dt", "dlat", "dlon"]] = 0'''

# change cog and heading to cos and sin for TCN
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])



df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205089000,22.032426,-77.320697,2023-03-01 06:50:00,13.6,107.8,107.0,1_205089000,2023-03-01 06:50:00,0.0,0.000000,0.000000,1.881465,0.952129,-0.305695,1.867502,0.956305,-0.292372
1,205089000,22.029460,-77.311284,2023-03-01 06:55:00,13.6,109.1,107.0,1_205089000,2023-03-01 06:55:00,300.0,-0.002966,0.009413,1.904154,0.944949,-0.327218,1.867502,0.956305,-0.292372
2,205089000,22.006908,-77.235489,2023-03-01 07:15:00,13.4,108.3,107.0,1_205089000,2023-03-01 07:15:00,1200.0,-0.022552,0.075795,1.890192,0.949425,-0.313992,1.867502,0.956305,-0.292372
3,205089000,21.999225,-77.208927,2023-03-01 07:20:00,13.4,106.0,106.0,1_205089000,2023-03-01 07:20:00,300.0,-0.007683,0.026562,1.850049,0.961262,-0.275637,1.850049,0.961262,-0.275637
4,205089000,21.989461,-77.173879,2023-03-01 07:30:00,13.4,106.8,107.0,1_205089000,2023-03-01 07:30:00,600.0,-0.009764,0.035048,1.864012,0.957319,-0.289032,1.867502,0.956305,-0.292372


In [6]:
# Reduce columns again
training_df = df[["MMSI", "voyage_id", "dt", "cog_sin", "cog_cos", "hdg_sin", "hdg_cos", "dlat", "dlon"]]

training_df.head()

,MMSI,voyage_id,dt,cog_sin,cog_cos,hdg_sin,hdg_cos,dlat,dlon
0,205089000,1_205089000,0.0,0.952129,-0.305695,0.956305,-0.292372,0.000000,0.000000
1,205089000,1_205089000,300.0,0.944949,-0.327218,0.956305,-0.292372,-0.002966,0.009413
2,205089000,1_205089000,1200.0,0.949425,-0.313992,0.956305,-0.292372,-0.022552,0.075795
3,205089000,1_205089000,300.0,0.961262,-0.275637,0.961262,-0.275637,-0.007683,0.026562
4,205089000,1_205089000,600.0,0.957319,-0.289032,0.956305,-0.292372,-0.009764,0.035048


In [7]:
# List of vessels
vessels = df['MMSI'].unique()

# Shuffle for randomness (VERY important)
rng = np.random.default_rng(42)
rng.shuffle(vessels)

n_total = len(vessels)

# ~90% train and ~10% testing
test_split = int(n_total * 0.9)
train_val_vessels = vessels[:test_split]
test_vessels      = vessels[test_split:]

# Split train into ~75% train and ~15% val
val_frac_of_train = 0.2   # ~20% of the ~90%
val_split = int(len(train_val_vessels) * (1 - val_frac_of_train))

train_vessels = train_val_vessels[:val_split]
val_vessels   = train_val_vessels[val_split:]

# Create DataFrames
df_train = df[df['MMSI'].isin(train_vessels)].copy()
df_val   = df[df['MMSI'].isin(val_vessels)].copy()
df_test  = df[df['MMSI'].isin(test_vessels)].copy()

# Diagnostics
print("Train %:", len(df_train)/len(df))
print("Val %:",   len(df_val)/len(df))
print("Test %:",  len(df_test)/len(df))

print("Train rows:", len(df_train))
print("Val rows:",   len(df_val))
print("Test rows:",  len(df_test))

Train %: 0.6734888148500714
Val %: 0.20022211645248295
Test %: 0.12628906869744566
Train rows: 33960
Val rows: 10096
Test rows: 6368


In [8]:
# More Diagnostics
print("number of Train vessels:", df_train["MMSI"].nunique())
print("number of Val vessels:", df_val["MMSI"].nunique())
print("number of Test vessels:", df_test["MMSI"].nunique())

print("number of Train voyages:", df_train["voyage_id"].nunique())
print("number of Val voyages:", df_val["voyage_id"].nunique())
print("number of Test voyages:", df_test["voyage_id"].nunique())

number of Train vessels: 777
number of Val vessels: 195
number of Test vessels: 109
number of Train voyages: 1926
number of Val voyages: 608
number of Test voyages: 396


In [9]:
# Create csv's
df_train.to_csv("mid_anom_train_data.csv", index=False)
df_val.to_csv("mid_anom_val_data.csv", index=False)
df_test.to_csv("mid_anom_test_data.csv", index=False)